# 2 · Dataset construction

From the two panels to the two datasets the models read. This is where the
**target** is defined, and where the look-ahead bias the paper measures is either
kept out or let in on purpose.

Each step is one call to a function of `src/preprocessing.py`, with what it does
above it. The command line does the same in one go:

```bash
python scripts/pipeline.py datasets
python scripts/pipeline.py datasets --example            # on the synthetic panels
python scripts/pipeline.py datasets --missing-threshold 0.4
```

## The two datasets

|  | `controlled` | `leakboth` |
|---|---|---|
| panel | timed | snapshot |
| attributes | of the row's own year | as declared at extraction time |
| features | read at the age the firm first reached an early stage | cumulated over its whole observed life |
| target | reaches the next stage **within T years** of that age | **ever** reaches the next stage |

Everything that makes the second one biased is in it at once, which is why the two
controls further down take that sum apart.

In [ ]:
import polars as pl
import yaml

from src.preprocessing import (
    FEATURE_COLUMNS,
    TARGET_COLUMNS,
    build_full_history_dataset,
    build_processed_datasets,
    build_windowed_dataset,
    preprocess_dataset,
)

# EXAMPLE reads the panels built from the synthetic extraction, and writes beside
# them: the released datasets cannot be overwritten by a demo run.
EXAMPLE = False

CONFIG = yaml.safe_load(open("config/config.yaml"))
T = int(CONFIG["T"])
LAST_YEAR = int(CONFIG["last_year"])
FIRST_DECISION_YEAR = int(CONFIG["first_decision_year"])
MAX_STARTING_AGE = int(CONFIG["max_starting_age"])
THRESHOLDS = CONFIG["preprocessing"]
RANKING = CONFIG["paths"]["raw_university_ranking"]

base = "data/example" if EXAMPLE else "data"
PANELS = {"timed": f"{base}/interim/panel_timed.csv.gz", "snapshot": f"{base}/interim/panel_snapshot.csv.gz"}
OUT = {
    "controlled": f"{base}/processed/dataset_controlled.csv",
    "leakboth": f"{base}/processed/dataset_leakboth.csv",
}

pl.Config.set_tbl_cols(10)
print("panels     :", PANELS["timed"], "and", PANELS["snapshot"])
print("horizon T:", T, "years, decisions from", FIRST_DECISION_YEAR, "up to", LAST_YEAR)
print("starting age at most:", MAX_STARTING_AGE)
print("thresholds :", THRESHOLDS)

---
## The two panels

They have to carry the same firm-years in the same order: the switches change
values, never rows. The two controls further down swap the target between the two
datasets on `CompanyID`, and that is only legitimate if this holds.

In [ ]:
timed = pl.read_csv(PANELS["timed"], null_values=["NA"])
snapshot = pl.read_csv(PANELS["snapshot"], null_values=["NA"])
keys = ["CompanyID", "Age"]
print(f"timed   : {timed.height:,} rows x {timed.width} columns")
print(f"snapshot: {snapshot.height:,} rows x {snapshot.width} columns")
print("same firm-years:", timed.select(keys).equals(snapshot.select(keys)))

The panel carries more than the models read: `TARGET_COLUMNS` are the columns that
place a firm in time and carry its outcome, `FEATURE_COLUMNS` are what the models
see. Everything else the panel holds is there for the construction, or declared as
an addition.

In [ ]:
print(f"{len(FEATURE_COLUMNS)} features, {len(TARGET_COLUMNS)} columns for the target")
print("carried but not read:", sorted(set(timed.columns) - set(FEATURE_COLUMNS) - set(TARGET_COLUMNS)))

---
## The bias-controlled dataset

**The sample.** Only the firms that were in an early stage at least once, and that
reached it **within `max_starting_age` years** of being founded (two, in `config.yaml`): the prediction is made from the
first early-stage year, so a firm that only reaches it at age five would be
predicted from a very different point in its life.

**The decision point** is that first early-stage year, and every feature is read
there. **The target** is read at the decision point plus the window, or at the
last year observed if the panel is shorter: does the firm reach the next stage
within T years? If the change happens later, at the decision point it is not
knowable, and the firm counts as not having moved.

**The calendar** matters too: only decision years from `first_decision_year` on, and only those that
leave a full window of future inside the extraction.

In [ ]:
windowed = build_windowed_dataset(timed, T, LAST_YEAR, FIRST_DECISION_YEAR, MAX_STARTING_AGE)
print(f"firms: {windowed.height:,}")
print(windowed["Target"].value_counts(sort=True))
windowed.select("CompanyID", "Age", "StartingAge", "TargetAge", "GrowthStageGroup", "Target").head(5)

**The preprocessing.** The institutes become one flag, "somebody from a top-50
university", because the raw column is a concatenation of names. The gender of the
chief executive becomes one indicator. The two categorical features are left as
raw categories on purpose: they are frequency-encoded **per split**, inside
`get_split`, so that a held-out row never contributes to its own encoding.

Then the missing-value policy. A feature missing on `missing_threshold` of the rows
or more is dropped: past that point the imputation would be inventing the column
rather than completing it. The per-row budget goes with it: the looser the column
threshold, the more sparse features stay in, and the more missing values an
ordinary row carries. Both numbers live in `config.yaml`, and both apply to this
dataset only.

Finally the target becomes binary: reaching a later stage or an exit is a one,
staying or failing is a zero.

In [ ]:
controlled = preprocess_dataset(
    windowed,
    RANKING,
    missing_threshold=THRESHOLDS["missing_threshold"],
    max_missing_per_row=THRESHOLDS["max_missing_per_row"],
)
dropped = sorted(set(FEATURE_COLUMNS) - set(controlled.columns))
print(f"dataset: {controlled.height:,} firms x {controlled.width} columns, prevalence {controlled['Target'].mean():.3f}")
print("features dropped by the missing-value threshold:", dropped)

---
## The dataset with the look-ahead

The same firms, described from the **snapshot** panel and over their whole observed
life: the amounts are summed, the shares and the indices averaged, and every other
column is taken at the last year on record. The target is whether the firm *ever*
reaches the next stage, with no horizon at all.

It is built on the firms of the windowed dataset, in the same order, and then
reduced to its columns, because every comparison of the paper assumes the two
carry the same firms described in two ways. It drops no row and no column for its
own missing values: a firm of the first that it cannot describe stops the build
with an error instead of disappearing.

In [ ]:
full_history = build_full_history_dataset(snapshot, controlled)
leakboth = preprocess_dataset(
    full_history,
    RANKING,
    flag_no_time_window=True,
    missing_threshold=THRESHOLDS["missing_threshold"],
    max_missing_per_row=THRESHOLDS["max_missing_per_row"],
)
leakboth = leakboth.filter(pl.col("CompanyID").is_in(controlled["CompanyID"].implode())).select(controlled.columns)
print(f"dataset: {leakboth.height:,} firms x {leakboth.width} columns, prevalence {leakboth['Target'].mean():.3f}")
print("same firms as the windowed one:", sorted(controlled["CompanyID"]) == sorted(leakboth["CompanyID"]))

The gap between the two prevalences is not noise: it is the label leak. The target
of the second is easier to reach, because "ever" has no deadline. No model here
tunes its decision threshold, so F1, precision and recall move with the prevalence
on their own, and **AUC is the metric to read across this axis**.

In [ ]:
comparison = pl.DataFrame(
    {
        "dataset": ["controlled", "leakboth"],
        "firms": [controlled.height, leakboth.height],
        "columns": [controlled.width, leakboth.width],
        "prevalence": [controlled["Target"].mean(), leakboth["Target"].mean()],
    }
)
comparison

---
## The same two datasets, in one call

Everything above is what `build_processed_datasets` does, and it is what the
command line calls: the steps are laid out here to be read, and composed there to
be run. The check below is the point, the two routes have to agree.

In [ ]:
controlled_2, leakboth_2 = build_processed_datasets(
    timed,
    snapshot,
    RANKING,
    T=T,
    last_year=LAST_YEAR,
    first_decision_year=FIRST_DECISION_YEAR,
    max_starting_age=MAX_STARTING_AGE,
    missing_threshold=THRESHOLDS["missing_threshold"],
    max_missing_per_row=THRESHOLDS["max_missing_per_row"],
)
print("same windowed dataset :", controlled.equals(controlled_2))
print("same full-history one :", leakboth.equals(leakboth_2))

In [ ]:
import pathlib

for name, dataset in (("controlled", controlled), ("leakboth", leakboth)):
    out = pathlib.Path(OUT[name])
    out.parent.mkdir(parents=True, exist_ok=True)
    dataset.write_csv(out)
    print("written:", out)

---
## Next

**`3_experiments.ipynb`** reads these two files, assembles the six experiments out
of them (the two datasets, the two ablations and the two controls), trains the
seven model families and compares them.